Verificar se dados estão no volume correto

In [0]:
%fs ls /Volumes/spark_definitive_guide/spark_book/activity-data/

Modo Batch

In [0]:
path = "/Volumes/spark_definitive_guide/spark_book/activity-data/"

static = spark.read.json(path)

dataSchema = static.schema

In [0]:
streaming = spark.readStream.schema(dataSchema).option("maxFilesPerTrigger", 1).json(path)

In [0]:
activityCounts = streaming.groupBy("gt").count()

In [0]:
#evita criar muitos shuffles partitions
spark.conf.set("spark.sql.shuffle.partitions", "5")

In [0]:
activityQuery = activityCounts.writeStream.trigger(availableNow=True).option("checkpointLocation", "/Volumes/spark_definitive_guide/spark_book/activity-data/_checkpoints/activity_counts").queryName("activity_counts").format("memory").outputMode("complete").start()


In [0]:
activityQuery.awaitTermination()

In [0]:
spark.streams.active

In [0]:
from time import sleep
for x in range(5):
    spark.sql("SELECT * FROM activity_counts").show()
    sleep(1)

In [0]:
from pyspark.sql.functions import expr
simpleTransform = streaming.withColumn("stairs", expr("gt like '%stairs%'"))\
    .where("stairs")\
    .where("gt is not null")\
    .select("gt", "model", "arrival_time", "creation_time")\
    .writeStream\
    .trigger(availableNow=True)\
    .option("checkpointLocation", "/Volumes/spark_definitive_guide/spark_book/activity-data/_checkpoints/simple_transform_v2")\
    .queryName("simple_transform")\
    .format("memory")\
    .outputMode("append")\
    .start()

In [0]:
from time import sleep
for x in range(5):
    spark.sql("SELECT * FROM simple_transform").show()
    sleep(1)

In [0]:
deviceModelStats = streaming.cube("gt", "model").avg()\
    .drop("avg(arrival_time)")\
    .drop("avg(creation_time)")\
    .drop("avg(index)")\
    .writeStream\
    .trigger(availableNow=True)\
    .option("checkpointLocation", "/Volumes/spark_definitive_guide/spark_book/activity-data/_checkpoints/device_counts")\
    .queryName("device_counts").format("memory")\
    .outputMode("complete")\
    .start()

In [0]:
%sql
select * from device_counts

In [0]:
historicalAgg = static.groupBy("gt", "model").avg()

deviceModelStats = streaming.drop("Arrival_Time", "Creation_Time", "Index")\
    .cube("gt", "model").avg()\
    .join(historicalAgg, ["gt", "model"])\
    .writeStream\
    .option("checkpointLocation", "/Volumes/spark_definitive_guide/spark_book/activity-data/_checkpoints/device_counts_join")\
    .queryName("device_counts").format("memory")\
    .outputMode("complete")\
    .trigger(availableNow=True)\
    .start()